In [1]:
import warnings
import sys

if not sys.warnoptions:
    warnings.simplefilter("ignore")
import os
import glob
import numpy as np
from scipy import stats

from nilearn import datasets, surface
from nilearn import plotting
from nilearn.image import resample_to_img, index_img
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.surface import SurfaceImage
import nibabel as nib

from brainiak import image, io
from brainiak.isc import isc, isfc, permutation_isc
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
 
from scipy import stats

sns.set(style="white", context="notebook", font_scale=1, rc={"lines.linewidth": 2})

In [2]:
from pathlib import Path
from tqdm import tqdm
 
from nilearn.maskers import NiftiMasker
from nilearn.image import resample_to_img, index_img
 
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
 
from scipy import stats

from scipy import stats
from joblib import Parallel, delayed
from sklearn.neighbors import KDTree

In [45]:
ROIS = {
    "Left Heschl's Gyrus":           ("destrieux",      ["L G_temp_sup-G_T_transv"]),
    "Left Superior Temporal Gyrus":      ("destrieux",      ["L G_temp_sup-Lateral"]),
    "Right Superior Temporal Gyrus":      ("destrieux",      ["R G_temp_sup-Lateral"]),
    "Planum temporale": ("destrieux", ["L G_temp_sup-Plan_tempo", "R G_temp_sup-Plan_tempo"]),
    "Left Superior Temporal Sulcus":       ("destrieux",      ["L S_temporal_sup"]),
    "Right Superior Temporal Sulcus":      ("destrieux", ["R S_temporal_sup"]),
    "Left Temporal pole":    ("destrieux", ["L Pole_temporal"]),
    "Right Temporal pole":    ("destrieux", ["R Pole_temporal"]),
    "Left Middle Frontal Gyrus": ("destrieux",      ["L G_front_middle"]),
    "Right Middle Frontal Gyrus": ("destrieux",      ["R G_front_middle"]),
    "Medial prefrontal cortex":        ("harvard_oxford", ["Frontal Medial Cortex", "Superior Frontal Gyrus"]),


    "Inferior Frontal Gyrus triangularis": ("destrieux",      ["L G_front_inf-Triangul",  "R G_front_inf-Triangul"]),
    "Inferior Frontal Gyrus opercularis":  ("destrieux",      ["L G_front_inf-Opercular", "R G_front_inf-Opercular"]),
    "Left Middle Temporal Gyrus":              ("destrieux", ["L G_temporal_middle"]),
    "Right Middle Temporal Gyrus":             ("destrieux", ["R G_temporal_middle"]),
    "Left Inferior Temporal Gyrus":            ("destrieux", ["L G_temporal_inf"]),
    "Right Inferior Temporal Gyrus":           ("destrieux", ["R G_temporal_inf"]),
    
    "Ventral striatum":            ("harvard_oxford_sub", ["Left Accumbens", "Right Accumbens"]),
    "Anterior Cingulate Cortex":  ("harvard_oxford", ["Cingulate Gyrus, anterior division"]),
    "Posterior Cingulate Cortex":  ("harvard_oxford", ["Cingulate Gyrus, posterior division"]),
    "Temporoparietal Junction":    ("harvard_oxford", ["Supramarginal Gyrus, anterior division","Supramarginal Gyrus, posterior division","Angular Gyrus"]), 
    "Precuneus":        ("destrieux",      ["L G_precuneus",           "R G_precuneus"]),
}

In [4]:
bids_dir = "/home/NEU480/datasets/narratives/"
fmriprep_dir = bids_dir + "derivatives/fmriprep/"
cleaned_dir = bids_dir + "derivatives/afni-nosmooth/"
stimuli_dir = bids_dir + "stimuli/"
transcript_dir =  "/home/vm5631/stimuli/transcripts_with_events/"
dir_nilearn = "/home/NEU480/datasets/nilearn_data"
code_dir = bids_dir + "code/"
net_id = os.environ['USER']
scratch_folder = f"/scratch/network/{net_id}/"
thesis_folder = scratch_folder + "thesis/"

In [5]:
import json
import pandas as pd


def json_to_events(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    events_list = []
    for event in data['annotations']['events']:
        events_list.append({
            'onset': event['start'],
            'duration': event['end'] - event['start'],
            'trial_type': event['label']  # e.g., 'punchline' or 'audience_laughter'
        })
    
    return pd.DataFrame(events_list)

In [6]:
def fetch_shared_punchlines(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    events_list = []
    for event in data['annotations']['events']:
        if 'shared_punchline_id' in event:
            events_list.append({
                'onset': event['start'],
                'duration': event['end'] - event['start'],
                'trial_type': event['label']  # e.g., 'punchline' or 'audience_laughter'
            })
    
    return pd.DataFrame(events_list)

In [7]:
events_live = json_to_events('stimuli/transcripts_with_events/pieman_audio.json')
events_live

,onset,duration,trial_type
0,69.218,2.582,punchline
1,71.954,1.546,audience_laughter
2,79.249,0.740,punchline
3,80.244,1.407,audience_laughter
4,86.093,2.561,punchline
5,88.592,2.128,audience_laughter
6,95.318,0.680,punchline
7,96.185,1.531,audience_laughter
8,115.150,1.421,punchline
9,131.451,0.861,punchline


In [8]:
events_pni = json_to_events('stimuli/transcripts_with_events/piemanpni_audio.json')
events_pni['onset'] = events_pni['onset'] + 12

In [9]:
events_pni

,onset,duration,trial_type
0,78.491,3.963,punchline
1,88.179,1.421,punchline
2,93.243,2.722,punchline
3,100.343,1.241,punchline
4,116.676,1.921,punchline
5,133.258,1.000,punchline
6,135.198,1.721,punchline
7,141.760,2.961,punchline
8,154.424,2.481,punchline
9,163.979,2.782,punchline


In [10]:
punchlines_live = events_live[events_live['trial_type'] == 'punchline'].reset_index(drop=True)
punchlines_live

,onset,duration,trial_type
0,69.218,2.582,punchline
1,79.249,0.740,punchline
2,86.093,2.561,punchline
3,95.318,0.680,punchline
4,115.150,1.421,punchline
5,131.451,0.861,punchline
6,135.898,2.345,punchline
7,143.037,2.682,punchline
8,156.448,1.901,punchline
9,164.053,1.902,punchline


In [11]:
punchlines_pni = events_pni[events_pni['trial_type'] == 'punchline'].reset_index(drop=True)
punchlines_pni

,onset,duration,trial_type
0,78.491,3.963,punchline
1,88.179,1.421,punchline
2,93.243,2.722,punchline
3,100.343,1.241,punchline
4,116.676,1.921,punchline
5,133.258,1.000,punchline
6,135.198,1.721,punchline
7,141.760,2.961,punchline
8,154.424,2.481,punchline
9,163.979,2.782,punchline


In [12]:
shared_punchlines_live = fetch_shared_punchlines('stimuli/transcripts_with_events/pieman_audio.json')
shared_punchlines_live

,onset,duration,trial_type
0,69.218,2.582,punchline
1,79.249,0.740,punchline
2,86.093,2.561,punchline
3,115.150,1.421,punchline
4,131.451,0.861,punchline
5,135.898,2.345,punchline
6,143.037,2.682,punchline
7,156.448,1.901,punchline
8,164.053,1.902,punchline
9,169.812,2.442,punchline


In [13]:
shared_punchlines_pni = fetch_shared_punchlines('stimuli/transcripts_with_events/piemanpni_audio.json')
shared_punchlines_pni['onset'] = shared_punchlines_pni['onset'] + 12

In [14]:
shared_punchlines_pni

,onset,duration,trial_type
0,78.491,3.963,punchline
1,88.179,1.421,punchline
2,93.243,2.722,punchline
3,116.676,1.921,punchline
4,133.258,1.000,punchline
5,135.198,1.721,punchline
6,141.760,2.961,punchline
7,154.424,2.481,punchline
8,163.979,2.782,punchline
9,169.142,2.861,punchline


In [15]:
tr = 1.5
n_scans_live = 300
n_scans_pni = 294
trim_front_live = 10
trim_back_live = 8
trim_front_pni = 14
trim_back_pni = 13
exclude_subjects_live = [1, 13, 14, 21, 22, 38, 56, 68, 69]
subs_live = [(i + 1) for i in range(16, 82)]
exclude_subjects_pni = [266, 268, 269, 270, 271, 280]
subs_pni = [(i + 1) for i in range(264, 315)]
frame_times_live = np.arange(n_scans_live) * tr
frame_times_pni = np.arange(n_scans_pni) * tr

In [16]:
def remove_subjects(sub_list, exclude_list):
    cleaned_list = sub_list
    for sub in exclude_list:
        if sub in sub_list:
            cleaned_list.remove(sub) 
    return cleaned_list

In [17]:
subs_live = remove_subjects(subs_live, exclude_subjects_live)

In [18]:
subs_pni = remove_subjects(subs_pni, exclude_subjects_pni)
subs_pni.append(127)

In [19]:
import random

def get_random_items(data_list, count):
    # random.sample raises an error if count > len(data_list)
    if count > len(data_list):
        return "Count is larger than the list size!"
    
    return random.sample(data_list, count)

In [20]:
live_random = get_random_items(subs_live, 45)
live_random

[52,
 79,
 37,
 25,
 33,
 49,
 61,
 71,
 29,
 42,
 19,
 66,
 74,
 23,
 26,
 48,
 70,
 34,
 27,
 59,
 57,
 75,
 53,
 67,
 46,
 81,
 62,
 20,
 60,
 45,
 41,
 31,
 24,
 55,
 30,
 36,
 80,
 18,
 63,
 32,
 72,
 17,
 28,
 39,
 51]

In [21]:
pni_random = get_random_items(subs_pni, 45)
pni_random

[304,
 288,
 306,
 315,
 293,
 297,
 290,
 292,
 279,
 300,
 282,
 308,
 276,
 283,
 311,
 267,
 287,
 295,
 272,
 296,
 302,
 285,
 265,
 291,
 274,
 273,
 275,
 299,
 303,
 127,
 281,
 307,
 312,
 294,
 278,
 284,
 286,
 301,
 298,
 289,
 310,
 309,
 305,
 313,
 314]

In [22]:
def get_data(subs, task):
    data = []
    for sub in subs:
        func_data = {}
        
        # Fit masker to extract mean time series for parcels
        file_name = f"sub-{sub:03d}_task-{task}_space-MNI152NLin2009cAsym_res-native_desc-clean_bold.nii.gz"
        file_dir = f"{cleaned_dir}sub-{sub:03d}/func/"
        func_fn = file_dir + file_name
        
        func_data['func'] = func_fn
        func_data['confounds'] = None
        func_data['id'] = sub
        data.append(func_data)
    return data

In [23]:
LIVE_PARTICIPANTS = get_data(live_random, "pieman")

In [24]:
STUDIO_PARTICIPANTS = get_data(pni_random, "piemanpni")

In [25]:
LIVE_ATLAS_PATHS = {
    "destrieux":      thesis_folder + "atlases/live/destrieux_resampled.nii.gz",
    "harvard_oxford": thesis_folder + "atlases/live/harvard_oxford_resampled.nii.gz",
    "harvard_oxford_sub": thesis_folder + "atlases/live/harvard_oxford_sub_resampled.nii.gz",
}
STUDIO_ATLAS_PATHS = {
    "destrieux":     thesis_folder + "atlases/studio/destrieux_resampled.nii.gz",
    "harvard_oxford": thesis_folder +"atlases/studio/harvard_oxford_resampled.nii.gz",
    "harvard_oxford_sub": thesis_folder + "atlases/studio/harvard_oxford_sub_resampled.nii.gz",
}

In [26]:
ATLAS_LABELS = {
    "destrieux":      {},
    "harvard_oxford": {},
    "harvard_oxford_sub": {},
}

In [37]:
"""
Surface MVPA Pipeline: Punchline vs Non-Punchline Classification
================================================================
Surface-based version of the MVPA pipeline. Uses GIFTI .func.gii
functional data, extracts per-trial LSS beta patterns within
Destrieux parcels on fsaverage5, classifies punchline vs baseline
with a linear SVM (LOO-CV), and compares accuracy between the live
and studio groups using a permutation test.

Key differences from the volumetric pipeline
---------------------------------------------
- Functional data is GIFTI (.func.gii), not NIfTI
- ROIs come from the Destrieux surface atlas (nilearn built-in,
  already on fsaverage5 — no volumetric mask resampling needed)
- GLM is run directly on vertex timeseries arrays using
  make_first_level_design_matrix + run_glm, bypassing
  FirstLevelModel (which is NIfTI-only)
- Analysis is run separately per hemisphere; results are merged
  at the group-comparison stage

Requirements: nilearn, nibabel, scikit-learn, scipy, numpy, pandas, joblib
"""
HRF_MODEL   = "spm"
N_PERMUTATIONS = 5000
N_JOBS         = -1   # -1 = all cores; set to e.g. 4 on a shared cluster

OUTPUT_DIR = Path("results/volumetric_mvpa")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFOUND_COLUMNS = None

In [28]:
def fetch_atlas_label_indices() -> dict:
    """
    Pull label_name -> integer_index dicts from nilearn for the two atlases.
    Run once; integer indices are the same regardless of atlas resolution.
    """
    from nilearn.datasets import fetch_atlas_destrieux_2009, fetch_atlas_harvard_oxford
 
    destrieux = fetch_atlas_destrieux_2009()
    destrieux_labels = {}
    for i, lbl in enumerate(destrieux.labels):
        name = lbl.decode() if isinstance(lbl, bytes) else lbl
        destrieux_labels[name] = i
 
    # Harvard-Oxford cortical maxprob atlas; label integer = list position.
    ho = fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
    ho_labels = {}
    for i, lbl in enumerate(ho.labels):
        name = lbl.decode() if isinstance(lbl, bytes) else lbl
        ho_labels[name] = i

    hos = fetch_atlas_harvard_oxford("sub-maxprob-thr25-2mm")
    hos_labels = {}
    for i, lbl in enumerate(hos.labels):
        name = lbl.decode() if isinstance(lbl, bytes) else lbl
        hos_labels[name] = i
 
    return {"destrieux": destrieux_labels, "harvard_oxford": ho_labels, "harvard_oxford_sub": hos_labels}

def resample_atlas_to_reference(atlas_path: str,
                                 reference_nifti: str,
                                 out_path: str) -> str:
    """
    Convenience helper if you haven't pre-resampled your atlases yet.
    Uses nearest-neighbour interpolation (label images must stay integer).
    """
    resampled = resample_to_img(
        source_img=atlas_path,
        target_img=reference_nifti,
        interpolation="nearest",
    )
    resampled.to_filename(out_path)
    return out_path

In [29]:
def build_roi_masks(atlas_paths: dict,
                    atlas_labels: dict,
                    roi_config: dict) -> dict:
    """
    Build {roi_name: 3D bool mask} for one group.
 
    Parameters
    ----------
    atlas_paths   : {atlas_name: path}              # resampled per group
    atlas_labels  : {atlas_name: {label_name: int}} # resolution-invariant
    roi_config    : {roi_name: (atlas_name, [label_names])}
    """
    # Load each atlas image once (integer array).
    atlas_data = {
        name: np.asarray(nib.load(path).get_fdata()).astype(int)
        for name, path in atlas_paths.items()
    }
 
    masks = {}
    for roi_name, (atlas_name, label_names) in roi_config.items():
        if atlas_name not in atlas_data:
            raise KeyError(
                f"ROI '{roi_name}' references atlas '{atlas_name}' which "
                f"was not provided in atlas_paths."
            )
        labels = atlas_labels[atlas_name]
        data   = atlas_data[atlas_name]
 
        combined = np.zeros(data.shape, dtype=bool)
        missing  = []
        for lbl in label_names:
            if lbl not in labels:
                missing.append(lbl)
                continue
            combined |= (data == labels[lbl])
 
        if missing:
            print(f"  WARNING: ROI '{roi_name}' — labels not found in "
                  f"{atlas_name}: {missing}")
        if not combined.any():
            print(f"  WARNING: ROI '{roi_name}' — mask is empty after lookup")
 
        masks[roi_name] = combined
    return masks

In [39]:
# =============================================================================
# 3. LOAD NIFTI FUNCTIONAL DATA AND BASELINE SAMPLING
# =============================================================================

def load_nifti_masked(nifti_path: str, mask_3d: np.ndarray) -> np.ndarray:
    """
    Load a 4D NIfTI and return a (n_voxels_in_mask, n_trs) array.
    `mask_3d` must match the NIfTI's spatial shape.
    """
    img = nib.load(nifti_path)
    if img.shape[:3] != mask_3d.shape:
        raise ValueError(
            f"NIfTI shape {img.shape[:3]} != mask shape {mask_3d.shape}. "
            f"Make sure the atlas was resampled to this participant's resolution."
        )
    data_4d = img.get_fdata()     # (x, y, z, t)
    return data_4d[mask_3d]       # (n_voxels_in_mask, n_trs)
 
 
def load_confounds(tsv_path, columns):
    if tsv_path is None or columns is None:
        return None
    df = pd.read_csv(tsv_path, sep="\t")
    available = [c for c in columns if c in df.columns]
    return df[available].fillna(0).values

def sample_baseline_onsets(punchline_df, func_duration, tr, n_samples,
                            min_gap_punchline=12.0, min_gap_self=4.0,
                            run_edge_buffer=20.0):
    punchline_onsets = punchline_df["onset"].values
    candidates = np.arange(run_edge_buffer, func_duration - run_edge_buffer, tr)
    far = np.array([
        t for t in candidates
        if np.all(np.abs(t - punchline_onsets) > min_gap_punchline)
    ])
    if len(far) < n_samples:
        raise ValueError(f"Not enough baseline candidates: {len(far)} < {n_samples}")
 
    rng = np.random.default_rng(42)
    shuffled = rng.permutation(far)
    chosen = []
    for t in shuffled:
        if not chosen or np.all(np.abs(t - np.array(chosen)) >= min_gap_self):
            chosen.append(t)
        if len(chosen) == n_samples:
            break
    if len(chosen) < n_samples:
        raise ValueError(f"Only found {len(chosen)} baseline onsets (need {n_samples})")
    return np.sort(chosen)

In [31]:
# =============================================================================
# 4. LSS ON SURFACE TIMESERIES
# =============================================================================

def build_lss_design_matrix(target_event, other_events, n_scans, tr):
    lss_events = pd.concat([
        target_event.assign(trial_type="target"),
        other_events.assign(trial_type="others"),
    ], ignore_index=True)
    frame_times = np.arange(n_scans) * tr
    return make_first_level_design_matrix(
        frame_times=frame_times,
        events=lss_events,
        hrf_model=HRF_MODEL,
        drift_model=None,
        high_pass=0,
    )
 
 
def lss_single_trial_beta(timeseries, target_event, other_events, confounds, tr):
    """
    timeseries : (n_features, n_scans) — voxels OR vertices
    Returns: (n_features,) beta estimates for the target regressor.
    """
    _, n_scans = timeseries.shape
    dm = build_lss_design_matrix(target_event, other_events, n_scans, tr)
    if confounds is not None:
        confound_df = pd.DataFrame(
            confounds,
            columns=[f"conf_{i}" for i in range(confounds.shape[1])],
        )
        dm = pd.concat([dm, confound_df.reset_index(drop=True)], axis=1)
 
    X = dm.values       # (n_scans, n_regressors)
    Y = timeseries.T    # (n_scans, n_features)
    betas, *_ = np.linalg.lstsq(X, Y, rcond=None)
    return betas[dm.columns.get_loc("target")]
 
 
def lss_beta_series(timeseries, events_df, confounds, tr, desc="LSS"):
    n_trials = len(events_df)
 
    def _one(i):
        target = events_df.iloc[[i]].copy()
        others = events_df.drop(index=i).reset_index(drop=True)
        return lss_single_trial_beta(timeseries, target, others, confounds, tr)
 
    betas = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(_one)(i) for i in tqdm(range(n_trials), desc=f"    {desc}", leave=False)
    )
    return np.vstack(betas)  # (n_trials, n_features)

In [32]:
# =============================================================================
# 6. ROI EXTRACTION AND CLASSIFICATION
# =============================================================================

def classify_loo(X_punchline, X_baseline):
    X = np.vstack([X_punchline, X_baseline])
    y = np.array([1] * len(X_punchline) + [0] * len(X_baseline))
    clf = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1))
    loo = LeaveOneOut()
    correct = [
        clf.fit(X[tr_], y[tr_]).predict(X[te])[0] == y[te][0]
        for tr_, te in loo.split(X)
    ]
    return float(np.mean(correct))

In [33]:
# =============================================================================
# 7. PER-PARTICIPANT MVPA (ONE HEMISPHERE)
# =============================================================================

def run_participant_mvpa(participant, events_df, tr, roi_masks, union_mask):
    """
    One participant, all ROIs. Returns {roi_name: accuracy}.
    """
    confounds = load_confounds(participant.get("confounds"), CONFOUND_COLUMNS)
    timeseries = load_nifti_masked(participant["func"], union_mask)
    n_trs        = timeseries.shape[1]
    run_duration = n_trs * tr
    median_dur   = float(events_df["duration"].median())
 
    print("    LSS — punchline trials...")
    punch_betas = lss_beta_series(timeseries, events_df, confounds, tr, desc="punchline")
 
    baseline_onsets = sample_baseline_onsets(
        events_df, run_duration, tr, n_samples=len(events_df)
    )
    baseline_events = pd.DataFrame({
        "onset":      baseline_onsets,
        "duration":   median_dur,
        "trial_type": "punchline",
    })
    print("    LSS — baseline trials...")
    base_betas = lss_beta_series(timeseries, baseline_events, confounds, tr, desc="baseline")
 
    accuracies = {}
    for roi_name, roi_mask in roi_masks.items():
        # Sub-index into the union-masked data: which entries belong to this ROI?
        sub_idx = roi_mask[union_mask]        # (n_union_voxels,) bool
        if sub_idx.sum() == 0:
            print(f"    WARNING: {roi_name} has 0 voxels; skipping.")
            accuracies[roi_name] = np.nan
            continue
        X_punch = punch_betas[:, sub_idx]
        X_base  = base_betas[:,  sub_idx]
        accuracies[roi_name] = classify_loo(X_punch, X_base)
    return accuracies

In [34]:
# =============================================================================
# 8. RUN ALL PARTICIPANTS IN A GROUP
# =============================================================================

def _run_one_participant(p, events_df, tr, roi_masks, union_mask, idx, total):
    label = p.get("func", f"participant_{idx}")
    print(f"\n  [{idx+1}/{total}] {Path(label).name}")
    accs = run_participant_mvpa(p, events_df, tr, roi_masks, union_mask)
    accs["participant"] = Path(label).name
    return accs
 
 
def run_group_mvpa(participants, events_df, tr, group_label, roi_masks):
    union_mask = np.logical_or.reduce(list(roi_masks.values()))
 
    print(f"\n{'='*60}")
    print(f"Group: {group_label}  |  {len(participants)} participants  |  TR={tr}s")
    print(f"Punchline events: {len(events_df)}  |  Union mask: {union_mask.sum():,} voxels")
    print(f"{'='*60}")
 
    rows = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(_run_one_participant)(p, events_df, tr, roi_masks, union_mask, i, len(participants))
        for i, p in enumerate(participants)
    )
    df = pd.DataFrame(rows)
    df["group"] = group_label
    return df

In [35]:
# =============================================================================
# 9. GROUP-LEVEL PERMUTATION TEST
# =============================================================================

def permutation_ttest(a, b, n_permutations=5000):
    observed_t, _ = stats.ttest_ind(a, b, equal_var=False)
    combined = np.concatenate([a, b])
    n_a = len(a)
    rng = np.random.default_rng(0)
    null_t = np.array([
        stats.ttest_ind((perm := rng.permutation(combined))[:n_a], perm[n_a:], equal_var=False)[0]
        for _ in range(n_permutations)
    ])
    return observed_t, float(np.mean(np.abs(null_t) >= np.abs(observed_t)))
 
 
def group_level_test(live_df, studio_df):
    meta_cols = {"participant", "group"}
    roi_cols  = [c for c in live_df.columns if c not in meta_cols]
 
    results = []
    print(f"\n{'='*60}")
    print(f"GROUP-LEVEL VOLUMETRIC MVPA RESULTS (permutation, n={N_PERMUTATIONS})")
    print(f"{'='*60}")
 
    for col in roi_cols:
        live_acc   = live_df[col].dropna().values
        studio_acc = studio_df[col].dropna().values
        if len(live_acc) < 3 or len(studio_acc) < 3:
            continue
 
        t_live,   p_live   = stats.ttest_1samp(live_acc,   0.5)
        t_studio, p_studio = stats.ttest_1samp(studio_acc, 0.5)
        t_btw,    p_btw    = permutation_ttest(live_acc, studio_acc, N_PERMUTATIONS)
 
        pooled_sd = np.sqrt(
            (np.std(live_acc, ddof=1)**2 + np.std(studio_acc, ddof=1)**2) / 2
        )
        cohens_d = ((np.mean(live_acc) - np.mean(studio_acc)) / pooled_sd
                    if pooled_sd > 0 else np.nan)
 
        sig = ("***" if p_btw < 0.001 else "**" if p_btw < 0.01
               else "*"  if p_btw < 0.05  else "ns")
 
        print(
            f"  {col:<30}  live={np.mean(live_acc):.3f}  "
            f"studio={np.mean(studio_acc):.3f}  "
            f"t={t_btw:+.3f}  p={p_btw:.4f} {sig}  d={cohens_d:+.3f}"
        )
 
        results.append({
            "ROI":                col,
            "live_mean_acc":      np.mean(live_acc),
            "live_sd":            np.std(live_acc, ddof=1),
            "live_vs_chance_p":   p_live,
            "studio_mean_acc":    np.mean(studio_acc),
            "studio_sd":          np.std(studio_acc, ddof=1),
            "studio_vs_chance_p": p_studio,
            "between_group_t":    t_btw,
            "between_group_p":    p_btw,
            "cohens_d":           cohens_d,
        })
 
    results_df = pd.DataFrame(results)
    from statsmodels.stats.multitest import multipletests
    _, p_fdr, _, _ = multipletests(results_df["between_group_p"], method="fdr_bh")
    results_df["between_group_p_fdr"] = p_fdr
    return results_df

In [46]:
# =============================================================================
# 10. MAIN
# =============================================================================

# --- Fetch label integer indices from nilearn (one-time, cached) ---
print("Fetching atlas label indices from nilearn...")
fetched = fetch_atlas_label_indices()
ATLAS_LABELS["destrieux"]      = fetched["destrieux"]
ATLAS_LABELS["harvard_oxford"] = fetched["harvard_oxford"]
ATLAS_LABELS["harvard_oxford_sub"] = fetched["harvard_oxford_sub"]

# --- Build ROI masks per group (each group uses its own resampled atlas) ---
print("\nBuilding ROI masks for LIVE group...")
live_masks = build_roi_masks(LIVE_ATLAS_PATHS, ATLAS_LABELS, ROIS)
print("\nBuilding ROI masks for STUDIO group...")
studio_masks = build_roi_masks(STUDIO_ATLAS_PATHS, ATLAS_LABELS, ROIS)

# --- Report voxel counts per ROI per group ---
print(f"\n{'ROI':<22} {'Live voxels':>12} {'Studio voxels':>15}")
print("-" * 52)
for roi_name in ROIS:
    lv = int(live_masks[roi_name].sum())   if roi_name in live_masks   else 0
    sv = int(studio_masks[roi_name].sum()) if roi_name in studio_masks else 0
    print(f"{roi_name:<22} {lv:>12,} {sv:>15,}")

# --- Run each group ---
live_acc_df = run_group_mvpa(
    LIVE_PARTICIPANTS, shared_punchlines_live, 1.5, "live", live_masks
)
studio_acc_df = run_group_mvpa(
    STUDIO_PARTICIPANTS, shared_punchlines_pni, 1.5, "studio", studio_masks
)

live_acc_df.to_csv(OUTPUT_DIR   / "live_volumetric_mvpa_accuracies.csv",   index=False)
studio_acc_df.to_csv(OUTPUT_DIR / "studio_volumetric_mvpa_accuracies.csv", index=False)

# --- Group-level test ---
results_df = group_level_test(live_acc_df, studio_acc_df)
results_df.to_csv(OUTPUT_DIR / "group_volumetric_mvpa_results.csv", index=False)
print(f"\nAll results saved to {OUTPUT_DIR}/")

Fetching atlas label indices from nilearn...


[fetch_atlas_destrieux_2009] Dataset found in /home/vm5631/nilearn_data/destrieux_2009

[fetch_atlas_harvard_oxford] Dataset found in /home/vm5631/nilearn_data/fsl

[fetch_atlas_harvard_oxford] Dataset found in /home/vm5631/nilearn_data/fsl


Building ROI masks for LIVE group...

Building ROI masks for STUDIO group...

ROI                     Live voxels   Studio voxels
----------------------------------------------------
Left Heschl's Gyrus              66             150
Left Superior Temporal Gyrus          306             717
Right Superior Temporal Gyrus          266             578
Planum temporale                153             366
Left Superior Temporal Sulcus          289             692
Right Superior Temporal Sulcus          362             871
Left Temporal pole              211             478
Right Temporal pole             240             556
Left Middle Frontal Gyrus          396             912
Right Middle Frontal Gyrus          386             900
Medial prefrontal cortex        1,339           3,179
Inferior Frontal Gyrus triangularis          195             439
Inferior Frontal Gyrus opercularis          333             802
Left Middle Temporal Gyrus          320             741
Right Middle Temporal 

...

    baseline:  27%|██▋       | 6/22 [00:00<00:00, 19.23it/s]  

    LSS — baseline trials...


    baseline:  55%|█████▍    | 12/22 [00:00<00:00, 17.69it/s]

    LSS — baseline trials...



GROUP-LEVEL VOLUMETRIC MVPA RESULTS (permutation, n=5000)
  Left Heschl's Gyrus             live=0.519  studio=0.546  t=-1.670  p=0.1056 ns  d=-0.352
  Left Superior Temporal Gyrus    live=0.560  studio=0.561  t=-0.033  p=1.0000 ns  d=-0.007
  Right Superior Temporal Gyrus   live=0.572  studio=0.563  t=+0.578  p=0.5686 ns  d=+0.122
  Planum temporale                live=0.571  studio=0.559  t=+0.772  p=0.4282 ns  d=+0.163
  Left Superior Temporal Sulcus   live=0.519  studio=0.537  t=-1.124  p=0.2752 ns  d=-0.237
  Right Superior Temporal Sulcus  live=0.535  studio=0.544  t=-0.541  p=0.6034 ns  d=-0.114
  Left Temporal pole              live=0.540  studio=0.536  t=+0.259  p=0.7836 ns  d=+0.055
  Right Temporal pole             live=0.538  studio=0.544  t=-0.400  p=0.6960 ns  d=-0.084
  Left Middle Frontal Gyrus       live=0.517  studio=0.543  t=-1.617  p=0.1156 ns  d=-0.341
  Right Middle Frontal Gyrus      live=0.531  studio=0.557  t=-1.507  p=0.1368 ns  d=-0.318
  Medial prefrontal c